In [40]:
print("hello World")

hello World


In [ ]:

from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("The_Courage_to_Be_Disliked.pdf")


data = loader.load()
page_contents = "\n".join(page.page_content for page in data)


print(data[50].metadata)

{'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-03-19T22:10:40+00:00', 'author': 'Ichiro Kishimi & Fumitake Koga', 'moddate': '2025-03-19T22:10:40+00:00', 'title': 'The Courage to Be Disliked', 'source': 'The_Courage_to_Be_Disliked.pdf', 'total_pages': 272, 'page': 50, 'page_label': '51'}


In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 1000
chunk_overlap = 200


splitter = RecursiveCharacterTextSplitter(
    separators=["\n"," ",""],
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap)



docs = splitter.split_text(page_contents)
print(docs[0])
print([len(doc) for doc in docs])

Thank you for downloading this Simon &
Schuster ebook.
Get a FREE ebook when you join our mailing list. Plus, get updates on new releases,
deals, recommended reads, and more from Simon & Schuster. Click below to sign up
and see terms and conditions.
CLICK HERE TO SIGN UP
Already a subscriber? Provide your email again so we can register this ebook and
send you more of what you like to read. You will continue to receive exclusive offers in
your inbox.
OceanofPDF.com
OceanofPDF.com
Contents
Authors’ Note
Introduction
THE FIRST NIGHT:
Deny Trauma
The Unknown Third Giant
Why People Can Change
Trauma Does Not Exist
People Fabricate Anger
How to Live Without Being Controlled by the Past
Socrates and Adler
Are You Okay Just As You Are?
Unhappiness Is Something You Choose for Yourself
People Always Choose Not to Change
Your Life Is Decided Here and Now
THE SECOND NIGHT:
All Problems Are Interpersonal Relationship Problems
Why You Dislike Yourself
[951, 966, 958, 940, 974, 988, 984, 928, 951, 97

In [59]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

vector = embedding_function.embed_query("hello world")
print(vector)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7484.52it/s]


[0.010723961517214775, 0.05578305199742317, 0.02708449587225914, 0.00304088881239295, 0.030335664749145508, 0.020480157807469368, 0.03132810816168785, 0.041038878262043, -0.025208471342921257, -0.057271808385849, -0.00396144762635231, -0.004360385704785585, -0.06811613589525223, 0.019528986886143684, 0.016956165432929993, 0.028180859982967377, 0.03159738704562187, 0.0007254044758155942, 0.015515279956161976, 0.03791613131761551, -0.05291657894849777, 0.009345244616270065, 0.03269641846418381, 0.015812644734978676, -0.006123300176113844, -0.007728766184300184, 0.001866715494543314, 0.04321492463350296, -0.09220389276742935, -0.005243562161922455, 0.02360629104077816, 0.0064338017255067825, 0.019542448222637177, -0.039408475160598755, 0.0038772744592279196, 0.023421097546815872, 0.0010543463286012411, 0.003118616994470358, -0.015606214292347431, -0.032448090612888336, -0.014722109772264957, -0.006371240131556988, -0.001469264505431056, 0.01682303287088871, -0.05387519672513008, -0.026068

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

vectorstore = Chroma.from_documents(
    [Document(page_content=chunk) for chunk in docs],
    embedding=embedding_function,
    persist_directory="VectorDatabase"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

message = """
Answer the following question using the context provided:

Context:
{context}

Question:
{question}

Answer:
"""

prompt_template = ChatPromptTemplate.from_messages([("human", message)])


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

import os
os.environ["GOOGLE_API_KEY"] = "key"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)



In [ ]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = ({"context":retriever , "question": RunnablePassthrough()}
            | prompt_template
            | llm)


response = rag_chain.invoke("who is a philosopher")
print(response.content)

Based on the context, a philosopher is one of the participants in the dialogue format that the book follows, conversing with a youth. The text also mentions that philosophers have been investigating themes like happiness, freedom, and the meaning of life since ancient Greek times.
